# Prepare Landsat data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import tarfile
from pathlib import Path

import rasterio as rio
import rioxarray  # noqa # Use to activate rio attributes

In [ ]:
from etdataset.api import (
    add_aux_data,
    create_dataset,
    download,
    download_aux,
    search,
)
from etdataset.dem import add_dem
from etdataset.plot import density_plot, plot_dataset, plot_dem, plot_images
from etdataset.provider import Collection
from etdataset.utils import dilate_mask, get_utm_bbox_from_roi
from etdataset.writer import (
    write_dataset,
    write_dataset_to_netcdf,
)

In [ ]:
import logging

from etdataset.logging import LoggerManager

LoggerManager.set_level(logging.DEBUG)

### Inputs

In [ ]:
# ROI
roi_path = os.path.join("data", "Zone_Senegal_Centre.shp")
roi_bbox, roi_crs = get_utm_bbox_from_roi(roi_path)
print(roi_bbox)
print(roi_crs)

In [ ]:
# Acquisition dates
min_date_str = "2023-03-01"  # Format YYYY-MM-DD
max_date_str = "2023-03-12"  # Format YYYY-MM-DD

In [ ]:
cloud_cover = 25
roi_coverage = 33

In [ ]:
output_path = "out"
os.makedirs(output_path, exist_ok=True)

In [ ]:
mnt_path = None
if os.environ.get("MNT_PATH", None) is not None:
    mnt_path = os.path.join(os.environ["MNT_PATH"], "DEM_Copernicus_30m/")

## Download

### Search products

In [ ]:
res = search(
    Collection.LANDSAT,
    min_date_str,
    max_date_str,
    roi_bbox=roi_bbox,
    roi_crs=roi_crs,
    max_cloud_cover=cloud_cover,
    min_roi_overlap=roi_coverage,
)

In [ ]:
# Keep only landsat 8
res = res[res["Product_name"].str.contains("LC08_", case=False, na=False)]

In [ ]:
res

### Download products

In [ ]:
download(products=res, output_dir=output_path)

### Donwload auxiliary data

In [ ]:
download_aux(products=res, output_dir=output_path)

## Prepare

### Extract archives

In [ ]:
# List archives
folder = Path(output_path) / "LANDSAT"
files = folder.glob("*.tar")  # Non-recursive

archives = list(folder.rglob("*.tar"))
archives

In [ ]:
# Untar archives
landsat_path = Path(output_path) / "LANDSAT"
for archive in archives:
    # Open and extract
    with tarfile.open(archive) as tar:
        extract_path = landsat_path / archive.stem
        if not extract_path.exists():
            tar.extractall(path=extract_path)

In [ ]:
# List products
landsat_path = Path(output_path) / "LANDSAT"
products = [f for f in landsat_path.iterdir() if f.is_dir()]
products

### Select one product

In [ ]:
product = "out/LANDSAT/LC08_L2SP_205050_20230303_20230316_02_T1"

### Prepare data from one product

#### Create dataset

In [ ]:
data = create_dataset(
    vis_path=product,
    roi_bbox=roi_bbox,
    roi_crs=roi_crs,
    resolution=90,
    resampling=rio.enums.Resampling.average,
)

In [ ]:
data

In [ ]:
plot_images(data)

Apply a dilation of cloud mask

In [ ]:
data["cloud"] = dilate_mask(data["cloud"], dilation=20)

In [ ]:
plot_images(data)

Add DEM data

In [ ]:
if mnt_path is not None:
    data = add_dem(data, mnt_dir=mnt_path)
    plot_dem(data[["height", "slope", "aspect"]])

In [ ]:
data

#### Add auxiliary data

In [ ]:
updated_data = add_aux_data(
    data=data,
    radiation_path=os.path.join(output_path, "MSG_data"),
    era5_path=os.path.join(output_path, "ERA5_data"),
)

In [ ]:
plot_dataset(updated_data)

In [ ]:
plot_dataset(updated_data)

In [ ]:
density_plot(updated_data)

#### Write data

In [ ]:
etdataset_path = Path(output_path) / "et_data"
os.makedirs(etdataset_path, exist_ok=True)

In [ ]:
write_dataset(xrds=updated_data, directory=etdataset_path, separate=False)

In [ ]:
write_dataset_to_netcdf(xrds=updated_data, directory=etdataset_path)